# Notebook 01 -- Baseline Model Testing (Group B: Forecasting Robustness)

**Research:** Hybrid AO-PSO Optimised Temporal Fusion Transformer for Smart Grid Multi-Horizon Energy Forecasting

---

## Purpose

This notebook executes the **10x Grind Protocol** for the three baseline models defined in the testing matrix (MASTER_BLUEPRINT.md, Group B):

| Model | Architecture | Role in Thesis |
|-------|-------------|----------------|
| Standard LSTM | Vanilla unidirectional, 2-layer stacked | Deep learning industry standard |
| Bi-LSTM | Bidirectional, forward/backward state fusion | Advanced sequence modelling standard |
| XGBoost | Gradient-boosted trees (MultiOutputRegressor) | Tree-based tabular standard |

Each model is trained and evaluated **10 independent times** on both the **Micro-Grid** (UCI Household) and **Macro-Grid** (ISO-NE) datasets. Per-run RMSE, MAE, and MAPE are recorded so that:

1. We report **Mean \u00b1 Std** performance for each baseline.
2. The per-run error vectors are saved to `results/baseline_metrics.json` for the **Wilcoxon signed-rank tests** in Notebook 04.

**Forecasting setup:** Window = 168h (7 days), Horizon = 24h, Chronological split (70/15/15), StandardScaler fitted on train only.

In [ ]:
# ================================================================
# Cell 1: Imports
# ================================================================
import sys
import os
import json
import time
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# ---- Set project root ----
PROJECT_ROOT = "/workspace/Hybrid-Optimizer-AO-PSO"

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), (
    f"src/ not found in {PROJECT_ROOT}. "
    f"Update PROJECT_ROOT to your actual project path on this machine."
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.dataset_utils import prepare_pipeline
from src.models import StandardLSTM, BiLSTM, XGBoostBaseline
from src.metrics import (
    inverse_transform_predictions,
    calculate_forecasting_metrics,
    calculate_horizon_metrics,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

In [ ]:
# ================================================================
# Cell 2: Configuration & Hyperparameters
# ================================================================

# --- 10x Grind Protocol ---
N_RUNS = 10

# --- Data pipeline ---
WINDOW_SIZE = 168        # 7 days of hourly data
HORIZON     = 24         # 24-hour forecast
BATCH_SIZE  = 64
CORE_COUNT = os.cpu_count() # Use all available CPU cores for data loading

# --- Deep learning baseline hyperparameters ---
DL_CONFIG = {
    "hidden_size": 128,
    "num_layers":  2,
    "dropout":     0.2,
    "epochs":      30,
    "lr":          1e-3,
}

# --- Dataset paths ---
MICRO_PATH = "data/micro_household_dataset.csv"
MACRO_PATH = "data/macro_grid_dataset.csv"

# --- Results directory ---
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Configuration")
print("=" * 40)
print(f"  N_RUNS      : {N_RUNS}")
print(f"  WINDOW_SIZE : {WINDOW_SIZE}")
print(f"  HORIZON     : {HORIZON}")
print(f"  BATCH_SIZE  : {BATCH_SIZE}")
print(f"  DL epochs   : {DL_CONFIG['epochs']}")
print(f"  DL lr       : {DL_CONFIG['lr']}")
print(f"  DL hidden   : {DL_CONFIG['hidden_size']}")
print(f"  Results dir : {RESULTS_DIR.resolve()}")

In [ ]:
# ================================================================
# Cell 3: Evaluation Wrapper
# ================================================================

def train_dl_model(model, train_loader, val_loader, epochs, lr, device):
    """
    Train a PyTorch model (LSTM or BiLSTM) with early stopping.

    Returns the trained model and the best validation loss.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    patience, patience_counter = 5, 0
    best_state = None

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        for x_cont, x_cat, y in train_loader:
            x_cont, y = x_cont.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(x_cont)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

        # --- Validate ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_cont, x_cat, y in val_loader:
                x_cont, y = x_cont.to(device), y.to(device)
                preds = model(x_cont)
                val_losses.append(criterion(preds, y).item())
        val_loss = np.mean(val_losses)

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_dl_model(model, test_loader, device):
    """Run inference on the test set, return (preds, targets) as numpy."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_cont, x_cat, y in test_loader:
            x_cont = x_cont.to(device)
            preds = model(x_cont).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y.numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)


def evaluate_baseline(model_class, pipeline, n_runs, is_xgboost=False):
    """
    Run the full 10x Grind evaluation for a single baseline model.

    Parameters
    ----------
    model_class : type
        StandardLSTM, BiLSTM, or XGBoostBaseline.
    pipeline : dict
        Output of prepare_pipeline().
    n_runs : int
        Number of independent runs.
    is_xgboost : bool
        If True, uses XGBoost-specific fit/predict path.

    Returns
    -------
    list[dict]
        Per-run metrics: [{"run": 1, "RMSE": ..., "MAE": ..., "MAPE": ...,
                           "horizon_metrics": {...}, "time_s": ...}, ...]
    """
    train_loader = pipeline["train_loader"]
    val_loader   = pipeline["val_loader"]
    test_loader  = pipeline["test_loader"]
    scaler       = pipeline["scaler"]
    target_idx   = pipeline["target_idx"]
    n_features   = pipeline["n_continuous"]
    model_name   = model_class.__name__

    results = []

    for run in range(1, n_runs + 1):
        t0 = time.time()

        if is_xgboost:
            # --- XGBoost path: collect all data into numpy ---
            model = model_class(horizon=HORIZON)

            # Collect train data
            X_train_list, y_train_list = [], []
            for x_cont, x_cat, y in train_loader:
                X_train_list.append(x_cont.numpy())
                y_train_list.append(y.numpy())
            X_train = np.concatenate(X_train_list)
            y_train = np.concatenate(y_train_list)

            # Collect test data
            X_test_list, y_test_list = [], []
            for x_cont, x_cat, y in test_loader:
                X_test_list.append(x_cont.numpy())
                y_test_list.append(y.numpy())
            X_test = np.concatenate(X_test_list)
            y_test = np.concatenate(y_test_list)

            model.fit(X_train, y_train)
            preds_scaled = model.predict(X_test)
            targets_scaled = y_test

        else:
            # --- PyTorch path: LSTM / BiLSTM ---
            model = model_class(
                n_features=n_features,
                hidden_size=DL_CONFIG["hidden_size"],
                num_layers=DL_CONFIG["num_layers"],
                dropout=DL_CONFIG["dropout"],
                horizon=HORIZON,
            )
            model = train_dl_model(
                model, train_loader, val_loader,
                epochs=DL_CONFIG["epochs"],
                lr=DL_CONFIG["lr"],
                device=DEVICE,
            )
            preds_scaled, targets_scaled = predict_dl_model(
                model, test_loader, DEVICE,
            )

        # --- Inverse transform to real-world units ---
        preds_real, targets_real = inverse_transform_predictions(
            preds_scaled, targets_scaled, scaler, target_idx,
        )

        # --- Calculate metrics ---
        overall = calculate_forecasting_metrics(targets_real, preds_real)
        per_horizon = calculate_horizon_metrics(targets_real, preds_real)

        elapsed = time.time() - t0

        run_result = {
            "run":  run,
            "RMSE": overall["RMSE"],
            "MAE":  overall["MAE"],
            "MAPE": overall["MAPE"],
            "horizon_metrics": {
                str(h): m for h, m in per_horizon.items()
            },
            "time_s": round(elapsed, 2),
        }
        results.append(run_result)

        print(f"  [{model_name}] Run {run:2d}/{n_runs} "
              f"| RMSE: {overall['RMSE']:.4f} "
              f"| MAE: {overall['MAE']:.4f} "
              f"| MAPE: {overall['MAPE']:.2f}% "
              f"| {elapsed:.1f}s")

        # --- Memory cleanup ---
        del model
        if not is_xgboost:
            torch.cuda.empty_cache()
        gc.collect()

    return results


def print_summary(model_name, results):
    """Print Mean +/- Std for RMSE, MAE, MAPE across all runs."""
    rmses = [r["RMSE"] for r in results]
    maes  = [r["MAE"]  for r in results]
    mapes = [r["MAPE"] for r in results]
    print(f"\n{'=' * 60}")
    print(f"  {model_name} -- {len(results)} runs")
    print(f"{'=' * 60}")
    print(f"  RMSE : {np.mean(rmses):.4f} \u00b1 {np.std(rmses):.4f}")
    print(f"  MAE  : {np.mean(maes):.4f} \u00b1 {np.std(maes):.4f}")
    print(f"  MAPE : {np.mean(mapes):.2f}% \u00b1 {np.std(mapes):.2f}%")
    print(f"{'=' * 60}")


print("Evaluation functions defined.")

In [ ]:
# ================================================================
# Cell 4: Micro-Grid Execution
# ================================================================
print("Loading Micro-Grid dataset (UCI Household)...")
micro_pipeline = prepare_pipeline(
    filepath=MICRO_PATH,
    dataset_type="micro",
    window_size=WINDOW_SIZE,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    num_workers=CORE_COUNT,
)
print(f"  n_continuous: {micro_pipeline['n_continuous']}")
print(f"  target_col:   {micro_pipeline['target_col']}")
print(f"  train batches: {len(micro_pipeline['train_loader'])}")
print(f"  test  batches: {len(micro_pipeline['test_loader'])}")

# --- LSTM ---
print("\n--- Standard LSTM (Micro-Grid) ---")
micro_lstm = evaluate_baseline(StandardLSTM, micro_pipeline, N_RUNS)
print_summary("LSTM (Micro)", micro_lstm)

# --- BiLSTM ---
print("\n--- Bi-LSTM (Micro-Grid) ---")
micro_bilstm = evaluate_baseline(BiLSTM, micro_pipeline, N_RUNS)
print_summary("Bi-LSTM (Micro)", micro_bilstm)

# --- XGBoost ---
print("\n--- XGBoost (Micro-Grid) ---")
micro_xgb = evaluate_baseline(XGBoostBaseline, micro_pipeline, N_RUNS, is_xgboost=True)
print_summary("XGBoost (Micro)", micro_xgb)

In [ ]:
# ================================================================
# Cell 5: Macro-Grid Execution
# ================================================================
print("Loading Macro-Grid dataset (ISO-NE)...")
macro_pipeline = prepare_pipeline(
    filepath=MACRO_PATH,
    dataset_type="macro",
    window_size=WINDOW_SIZE,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    num_workers=CORE_COUNT,
)
print(f"  n_continuous: {macro_pipeline['n_continuous']}")
print(f"  target_col:   {macro_pipeline['target_col']}")
print(f"  train batches: {len(macro_pipeline['train_loader'])}")
print(f"  test  batches: {len(macro_pipeline['test_loader'])}")

# --- LSTM ---
print("\n--- Standard LSTM (Macro-Grid) ---")
macro_lstm = evaluate_baseline(StandardLSTM, macro_pipeline, N_RUNS)
print_summary("LSTM (Macro)", macro_lstm)

# --- BiLSTM ---
print("\n--- Bi-LSTM (Macro-Grid) ---")
macro_bilstm = evaluate_baseline(BiLSTM, macro_pipeline, N_RUNS)
print_summary("Bi-LSTM (Macro)", macro_bilstm)

# --- XGBoost ---
print("\n--- XGBoost (Macro-Grid) ---")
macro_xgb = evaluate_baseline(XGBoostBaseline, macro_pipeline, N_RUNS, is_xgboost=True)
print_summary("XGBoost (Macro)", macro_xgb)

In [ ]:
# ================================================================
# Cell 6: Save Results
# ================================================================
# Structure: { dataset -> { model -> [ {run, RMSE, MAE, MAPE, ...} ] } }
# This is the exact format Notebook 04 expects for Wilcoxon tests.

all_results = {
    "micro": {
        "StandardLSTM": micro_lstm,
        "BiLSTM":       micro_bilstm,
        "XGBoost":      micro_xgb,
    },
    "macro": {
        "StandardLSTM": macro_lstm,
        "BiLSTM":       macro_bilstm,
        "XGBoost":      macro_xgb,
    },
    "config": {
        "n_runs":      N_RUNS,
        "window_size": WINDOW_SIZE,
        "horizon":     HORIZON,
        "batch_size":  BATCH_SIZE,
        "dl_config":   DL_CONFIG,
    },
}

output_path = RESULTS_DIR / "baseline_metrics.json"
with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print("\n>>> BASELINE TESTING COMPLETE <<<")